# NFL Roster Construction & Cap Efficiency
### 02: Efficiency Analysis

Merges team cap allocation with team performance, converts allocation to
% of total cap per position group, and tests whether *how* a team spends
predicts wins/point differential better than *how much* it spends.

In [26]:
import pandas as pd
import numpy as np

team_cap_allocation = pd.read_csv('../data/team_cap_allocation.csv')
team_performance = pd.read_csv('../data/team_performance.csv')

print(f"Cap allocation shape: {team_cap_allocation.shape}")
print(f"Team performance shape: {team_performance.shape}")

Cap allocation shape: (3436, 4)
Team performance shape: (384, 5)


### Standardize team abbreviations

Several franchises changed team codes during this window (relocations):
OAK -> LV (Raiders, 2020), SD -> LAC (Chargers, 2017), STL -> LA (Rams, 2016).
If left unstandardized, these show up as different teams in the two datasets
and cause a many-to-many merge (duplicate rows) instead of a clean one-to-one
join. Standardizing to each team's current code before merging fixes this.

In [27]:
team_code_fixes = {
    'OAK': 'LV',
    'SD': 'LAC',
    'STL': 'LA',
}

team_cap_allocation['team'] = team_cap_allocation['team'].replace(team_code_fixes)
team_performance['team'] = team_performance['team'].replace(team_code_fixes)

print("Unique teams in cap allocation:", team_cap_allocation['team'].nunique())
print("Unique teams in performance:", team_performance['team'].nunique())

Unique teams in cap allocation: 37
Unique teams in performance: 32


### Pivot to wide format

Turn the long cap-allocation table (one row per team/season/position group)
into one row per team-season, with a column per position group. This is
the shape needed to compare allocation *patterns* across position groups.

In [28]:
cap_wide = team_cap_allocation.pivot_table(
    index=['team', 'season'],
    columns='position_group',
    values='total_cap_allocated',
    aggfunc='sum',
    fill_value=0
).reset_index()

cap_wide.columns.name = None
print(f"Wide cap table shape: {cap_wide.shape}")
cap_wide.head()

Wide cap table shape: (384, 11)


,team,season,CB_S,DL,LB,OL,QB,RB,ST,TE,WR
0,ARI,2016,47.164287,25.541979,6.883084,27.289530,25.315000,3.301091,4.0380,6.095825,15.493175
1,ARI,2017,41.372422,13.267494,10.192540,28.016685,27.227400,7.345287,8.2375,10.600625,17.512047
2,ARI,2018,33.790741,32.143043,10.995940,26.971332,28.528640,17.553576,6.1667,10.251600,20.961408
3,ARI,2019,34.266140,20.601633,9.958000,32.406618,11.251994,16.959404,3.3027,4.110000,23.262409
4,ARI,2020,49.165780,24.781836,27.090771,39.564858,10.437161,11.649851,5.7486,7.592966,43.922048


In [29]:
# Convert each position group to % of that team-season's total cap
position_cols = [c for c in cap_wide.columns if c not in ['team', 'season']]

cap_wide['total_cap'] = cap_wide[position_cols].sum(axis=1)

for col in position_cols:
    cap_wide[f'pct_{col}'] = cap_wide[col] / cap_wide['total_cap']

pct_cols = [f'pct_{col}' for col in position_cols]
cap_wide[['team', 'season', 'total_cap'] + pct_cols].head()

,team,season,total_cap,pct_CB_S,pct_DL,pct_LB,pct_OL,pct_QB,pct_RB,pct_ST,pct_TE,pct_WR
0,ARI,2016,161.121971,0.292724,0.158526,0.042720,0.169372,0.157117,0.020488,0.025062,0.037834,0.096158
1,ARI,2017,163.772000,0.252622,0.081012,0.062236,0.171071,0.166252,0.044851,0.050299,0.064728,0.106929
2,ARI,2018,187.362980,0.180349,0.171555,0.058688,0.143952,0.152264,0.093688,0.032913,0.054715,0.111876
3,ARI,2019,156.118898,0.219487,0.131961,0.063785,0.207577,0.072073,0.108631,0.021155,0.026326,0.149004
4,ARI,2020,219.953871,0.223528,0.112668,0.123166,0.179878,0.047452,0.052965,0.026135,0.034521,0.199688


### Merge with team performance

In [30]:
master = pd.merge(cap_wide, team_performance, on=['team', 'season'], how='inner')
master['win_pct'] = master['wins'] / master['games']

print(f"Master analysis table shape: {master.shape}")
master[['team', 'season', 'total_cap', 'wins', 'win_pct', 'point_diff']].head()

Master analysis table shape: (369, 25)


,team,season,total_cap,wins,win_pct,point_diff
0,ARI,2016,161.121971,7,0.4375,56.0
1,ARI,2017,163.772000,8,0.5000,-66.0
2,ARI,2018,187.362980,3,0.1875,-200.0
3,ARI,2019,156.118898,5,0.3125,-81.0
4,ARI,2020,219.953871,8,0.5000,43.0


### Sanity check: confirm one row per team-season

After standardizing team codes, each team-season should appear exactly once.
If the row count and unique team-season count don't match, there are still
duplicate rows from the merge and the position-group counts downstream
will be inflated and unreliable.

In [31]:
print(f"Master rows: {len(master)}")
print(f"Unique team-season pairs: {master[['team','season']].drop_duplicates().shape[0]}")

if len(master) != master[['team','season']].drop_duplicates().shape[0]:
    print("\n⚠️ Duplicates still present — inspect below:")
    dupes = master[master.duplicated(subset=['team','season'], keep=False)]
    print(dupes[['team','season']].sort_values(['team','season']))
else:
    print("\n✅ Clean — one row per team-season.")

Master rows: 369
Unique team-season pairs: 369

✅ Clean — one row per team-season.


### Does allocation pattern predict performance better than total spend?

Two quick comparisons:
1. Correlation of **total cap spend** with win_pct / point_diff (the naive "spend more, win more" hypothesis)
2. Correlation of **allocation percentages by position group** with win_pct / point_diff (does *where* the money goes matter more than *how much*)

In [32]:
print("Correlation: total cap spend vs performance")
print(master[['total_cap', 'win_pct', 'point_diff']].corr()[['win_pct', 'point_diff']].loc[['total_cap']])

print("\nCorrelation: position-group allocation % vs performance")
corr_table = master[pct_cols + ['win_pct', 'point_diff']].corr()[['win_pct', 'point_diff']].drop(['win_pct', 'point_diff'])
corr_table = corr_table.sort_values('win_pct', ascending=False)
corr_table

Correlation: total cap spend vs performance
            win_pct  point_diff
total_cap  0.107259    0.123797

Correlation: position-group allocation % vs performance


,win_pct,point_diff
pct_QB,0.082897,0.066826
pct_TE,0.071538,0.063965
pct_CB_S,0.056250,0.083443
pct_ST,0.049318,0.053844
pct_WR,0.024066,0.024614
pct_RB,-0.022036,-0.017097
pct_OL,-0.068123,-0.095894
pct_LB,-0.079832,-0.074531
pct_DL,-0.080506,-0.069181


### Regression: allocation pattern vs. total spend alone

Compare how much variance in win_pct is explained by total cap spend alone,
versus how much is explained once position-group allocation % is added.

**Important:** the pct_ columns all sum to 1 for every row, since they are
shares of the same total. Including all of them plus total_cap creates
near-perfect multicollinearity and produces unstable, unreliable
coefficients. One position group is dropped as the baseline before fitting —
standard practice for compositional data. Its effect is captured implicitly
through the other coefficients (each one reads as "relative to the baseline
group").

In [33]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Drop one position group as baseline to avoid multicollinearity
# (pct columns sum to 1, so one must be excluded)
baseline_group = 'pct_ST'
pct_cols_model = [c for c in pct_cols if c != baseline_group]

# Model 1: total cap spend only
X1 = master[['total_cap']]
y = master['win_pct']

model1 = LinearRegression().fit(X1, y)
r2_spend_only = r2_score(y, model1.predict(X1))

# Model 2: total cap spend + position-group allocation % (baseline dropped)
X2 = master[['total_cap'] + pct_cols_model]
model2 = LinearRegression().fit(X2, y)
r2_with_allocation = r2_score(y, model2.predict(X2))

print(f"Baseline group (excluded from model): {baseline_group}")
print(f"R² (total spend only):              {r2_spend_only:.3f}")
print(f"R² (spend + position allocation %): {r2_with_allocation:.3f}")
print(f"Improvement from allocation pattern: {r2_with_allocation - r2_spend_only:.3f}")

Baseline group (excluded from model): pct_ST
R² (total spend only):              0.012
R² (spend + position allocation %): 0.040
Improvement from allocation pattern: 0.029


### Which position groups matter most?

Coefficients from Model 2 show which allocation percentages are most
associated with winning, holding total spend constant. Each coefficient is
relative to the excluded baseline group (Special Teams).

In [34]:
coef_table = pd.DataFrame({
    'feature': X2.columns,
    'coefficient': model2.coef_
}).sort_values('coefficient', ascending=False)

coef_table

,feature,coefficient
7,pct_TE,0.306878
0,total_cap,0.000373
1,pct_CB_S,-0.072285
6,pct_RB,-0.076537
5,pct_QB,-0.078487
8,pct_WR,-0.088958
2,pct_DL,-0.419217
4,pct_OL,-0.439612
3,pct_LB,-0.508162


### Efficiency outliers: who's spending well, who isn't

For each position group, flag team-seasons that spent a high % of cap but
underperformed (win_pct below league median), and teams that spent a low %
but overperformed — the "doing more with less" story.

In [35]:
league_median_win_pct = master['win_pct'].median()

outlier_rows = []
for col in pct_cols:
    pos = col.replace('pct_', '')
    high_spend_thresh = master[col].quantile(0.75)
    low_spend_thresh = master[col].quantile(0.25)

    overspend_underperform = master[(master[col] >= high_spend_thresh) & (master['win_pct'] < league_median_win_pct)]
    underspend_overperform = master[(master[col] <= low_spend_thresh) & (master['win_pct'] > league_median_win_pct)]

    outlier_rows.append({
        'position_group': pos,
        'overspend_underperform_count': len(overspend_underperform),
        'underspend_overperform_count': len(underspend_overperform)
    })

pd.DataFrame(outlier_rows)

,position_group,overspend_underperform_count,underspend_overperform_count
0,CB_S,35,40
1,DL,49,43
2,LB,45,51
3,OL,45,43
4,QB,42,49
5,RB,46,43
6,ST,38,39
7,TE,36,45
8,WR,42,36


In [36]:
# Example: pull the specific team-seasons for one position group of interest
# (change 'pct_OL' to any position group column to inspect)
position_of_interest = 'pct_OL'

print(f"Teams that spent heavily on {position_of_interest} but underperformed:")
high_thresh = master[position_of_interest].quantile(0.75)
display_cols = ['team', 'season', position_of_interest, 'win_pct', 'point_diff']
print(master[(master[position_of_interest] >= high_thresh) & (master['win_pct'] < league_median_win_pct)][display_cols].sort_values('win_pct'))

print(f"\nTeams that spent little on {position_of_interest} but overperformed:")
low_thresh = master[position_of_interest].quantile(0.25)
print(master[(master[position_of_interest] <= low_thresh) & (master['win_pct'] > league_median_win_pct)][display_cols].sort_values('win_pct', ascending=False))

Teams that spent heavily on pct_OL but underperformed:
    team  season    pct_OL   win_pct  point_diff
79   CLE    2017  0.308322  0.000000      -176.0
52   CAR    2023  0.232003  0.117647      -180.0
346  TEN    2014  0.250203  0.125000      -184.0
280  NYJ    2020  0.263052  0.125000      -214.0
272  NYG    2024  0.261501  0.176471      -142.0
164  JAX    2021  0.252008  0.176471      -204.0
63   CHI    2022  0.244634  0.176471      -137.0
141  HOU    2022  0.251858  0.176471      -131.0
119  DET    2021  0.303302  0.176471      -142.0
357  WAS    2013  0.311920  0.187500      -144.0
57   CHI    2016  0.261460  0.187500      -120.0
281  NYJ    2021  0.231272  0.235294      -194.0
153  IND    2022  0.251136  0.235294      -138.0
206   LV    2018  0.293523  0.250000      -177.0
274  NYJ    2014  0.295407  0.250000      -118.0
267  NYG    2019  0.236846  0.250000      -110.0
333   TB    2013  0.294598  0.250000      -101.0
358  WAS    2014  0.268711  0.250000      -137.0
53   CAR    20

In [37]:
# Save the final analysis table for use in Tableau / further analysis
master.to_csv('../data/team_cap_efficiency_master.csv', index=False)
print("Saved team_cap_efficiency_master.csv")

Saved team_cap_efficiency_master.csv
